## Importar librerías e instalar paquetes

In [1]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForImageClassification, AutoFeatureExtractor

from torchvision import transforms
from tqdm import tqdm
import sklearn.metrics as metrics
import pandas as pd

## Usar GPU

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", DEVICE)

Usando dispositivo: cuda


## Características de cada modelo

In [3]:
model_data = {
    "swinv2": {
        "image_size": 256,
        "model_name": "mirluvams/swinv2-base-patch4-window16-256-popocatepetl"
    },
    "beit": {
        "image_size": 224,
        "model_name": "mirluvams/beit-base-patch16-224-popocatepetl"
    },
    "vit": {
        "image_size": 224,
        "model_name": "mirluvams/vit-base-patch16-224-popocatepetl"
    },
    "convnext": {
        "image_size": 224,
        "model_name": "mirluvams/convnext-base-224-popocatepetl"
    },
    "dinov2": {
        "image_size": 224,
        "model_name": "mirluvams/dinov2-base-popocatepetl"
    }
}

In [4]:
#model_data["swinv2"]["model_name"]

## Cargar dataset de prueba

In [5]:
dataset = load_dataset("mirluvams/popocatepetl")
test_dataset = dataset["test"]

README.md:   0%|          | 0.00/608 [00:00<?, ?B/s]

data/train-00000-of-00011.parquet:   0%|          | 0.00/488M [00:00<?, ?B/s]

data/train-00001-of-00011.parquet:   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00002-of-00011.parquet:   0%|          | 0.00/479M [00:00<?, ?B/s]

data/train-00003-of-00011.parquet:   0%|          | 0.00/482M [00:00<?, ?B/s]

...

data/train-00010-of-00011.parquet:   0%|          | 0.00/479M [00:00<?, ?B/s]

data/test-00000-of-00002.parquet:   0%|          | 0.00/297M [00:00<?, ?B/s]

data/test-00001-of-00002.parquet:   0%|          | 0.00/296M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/216352 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24040 [00:00<?, ? examples/s]

In [6]:
print("Columnas:", test_dataset.column_names)
print("Cant. de imgs de test:", len(test_dataset))

Columnas: ['label', 'image', 'video', 'id']
Cant. de imgs de test: 24040


In [7]:
dataset["test"].set_format("torch")

In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 216352
    })
    test: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 24040
    })
})

## Función de evaluación de los modelos

In [9]:
BATCH_SIZE=16
def evaluate_model(model_name, image_size, **kwargs):
    print(f"\nEvaluando modelo: {model_name}")
    
    # Cargar modelo
    model = AutoModelForImageClassification.from_pretrained(model_name).to(DEVICE)
    
    # Transformar datos
    _test_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225]),
    ])
    
    def test_transform(ex):
        ex["pixel_values"] = [_test_transform(image) for image in ex["image"]]
        return ex
   
    dataset["test"].set_transform(test_transform)

    # Evaluar
    preds, labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(dataset["test"].iter(BATCH_SIZE), desc="Evaluando", total=len(dataset["test"])//BATCH_SIZE+1):
            pixel_values = torch.stack(batch["pixel_values"]).to(DEVICE)
            batch_labels = batch["label"]
            
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            
            preds.extend(predictions.cpu().numpy())
            labels.extend(batch_labels)
    
    # Calcular métricas
    acc = metrics.accuracy_score(labels, preds)
    f1 = metrics.f1_score(labels, preds, average="weighted")
    f1_micro = metrics.f1_score(labels, preds, average="micro")
    f1_macro = metrics.f1_score(labels, preds, average="macro")
    
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | F1_micro: {f1_micro:.4f} | F1_macro: {f1_macro:.4f}")
    return {"model": model_name, "accuracy": acc, "f1": f1, "f1_micro": f1_micro, "f1_macro": f1_macro}

## Resultados de evaluación de los modelos

In [10]:
#evaluate_model(model_data["swinv2"]["model_name"], model_data["swinv2"]["image_size"])
#for model in model_data.keys():
#   evaluate_model(**model_data["swinv2"])

In [11]:
results = []

In [12]:
for model in model_data.keys():
    result = evaluate_model(**model_data[model])
    results.append(result)


Evaluando modelo: mirluvams/swinv2-base-patch4-window16-256-popocatepetl


config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/348M [00:00<?, ?B/s]

Evaluando: 100%|████████████████████████████████████████████████████████████████████| 1503/1503 [08:20<00:00,  3.00it/s]


Accuracy: 0.9894 | F1: 0.9894 | F1_micro: 0.9894 | F1_macro: 0.9864

Evaluando modelo: mirluvams/beit-base-patch16-224-popocatepetl


...

Accuracy: 0.9869 | F1: 0.9869 | F1_micro: 0.9869 | F1_macro: 0.9832

Evaluando modelo: mirluvams/dinov2-base-popocatepetl


config.json:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Evaluando: 100%|████████████████████████████████████████████████████████████████████| 1503/1503 [07:15<00:00,  3.45it/s]

Accuracy: 0.9786 | F1: 0.9787 | F1_micro: 0.9786 | F1_macro: 0.9725


In [15]:
df_results_finetuned = pd.DataFrame(results)

In [16]:
df_results_finetuned

,model,accuracy,f1,f1_micro,f1_macro
0,mirluvams/swinv2-base-patch4-window16-256-popo...,0.989434,0.989440,0.989434,0.986394
1,mirluvams/beit-base-patch16-224-popocatepetl,0.986273,0.986286,0.986273,0.982046
2,mirluvams/vit-base-patch16-224-popocatepetl,0.987438,0.987449,0.987438,0.983497
3,mirluvams/convnext-base-224-popocatepetl,0.986897,0.986932,0.986897,0.983164
4,mirluvams/dinov2-base-popocatepetl,0.978577,0.978651,0.978577,0.972463


In [20]:
df_results_finetuned.to_csv('results_finetuned.csv')